# 🌾 AgriLens — Crop Disease Detection Model Training

This notebook trains a CNN (using transfer learning with MobileNetV2) to detect crop diseases from leaf images.

**Run this in Google Colab** (free GPU): https://colab.research.google.com

Steps:
1. Download PlantVillage dataset
2. Preprocess & augment images
3. Build CNN with transfer learning
4. Train & evaluate
5. Export model for Flask API

## 1. Setup — Enable GPU
Go to: Runtime → Change runtime type → Hardware accelerator → **GPU (T4)**

In [ ]:
!pip install tensorflow kagglehub -q

import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Download PlantVillage Dataset

In [ ]:
import kagglehub

# Downloads the PlantVillage dataset (~54,000 images, 38 classes)
dataset_path = kagglehub.dataset_download('emmarex/plantdisease')
print('Dataset downloaded to:', dataset_path)

import os
for root, dirs, files in os.walk(dataset_path):
    if files:
        print(root, '->', len(files), 'images')
        break

## 3. Prepare Data

**Tip for your final year report:** To keep training fast and your project scope manageable, start with 5-8 crop classes (e.g., Tomato — Healthy, Early Blight, Late Blight, Leaf Mold, Bacterial Spot) instead of all 38. You can mention in your report that the architecture scales to the full dataset.

In [ ]:
import os

PLANTVILLAGE_DIR = dataset_path  # adjust if nested folder

# List all available classes so you can pick which ones to use
all_classes = sorted(os.listdir(PLANTVILLAGE_DIR))
print(f'Found {len(all_classes)} classes:')
for c in all_classes:
    print(' -', c)

In [ ]:
# CHOOSE YOUR CLASSES HERE — start small, expand later
# Example: focus on Tomato diseases (common in Tamil Nadu)
SELECTED_CLASSES = [
    'Tomato_healthy',
    'Tomato_Early_blight',
    'Tomato_Late_blight',
    'Tomato_Leaf_Mold',
    'Tomato_Bacterial_spot',
]
# NOTE: match these exactly to folder names printed above — adjust as needed

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
import shutil

# Build a filtered dataset folder with only selected classes
FILTERED_DIR = '/content/filtered_dataset'
os.makedirs(FILTERED_DIR, exist_ok=True)

for cls in SELECTED_CLASSES:
    src = os.path.join(PLANTVILLAGE_DIR, cls)
    dst = os.path.join(FILTERED_DIR, cls)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
        print(f'Copied {cls}: {len(os.listdir(dst))} images')

## 4. Data Augmentation & Loading

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    validation_split=0.2,
)

train_gen = train_datagen.flow_from_directory(
    FILTERED_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
)

val_gen = train_datagen.flow_from_directory(
    FILTERED_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
)

CLASS_NAMES = list(train_gen.class_indices.keys())
print('Classes:', CLASS_NAMES)
NUM_CLASSES = len(CLASS_NAMES)

## 5. Build Model — Transfer Learning with MobileNetV2

We use MobileNetV2 (pretrained on ImageNet) as a feature extractor, then add our own classification head. This is much faster to train than building a CNN from scratch, and works well even on a free Colab GPU.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # freeze pretrained layers initially

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## 6. Train the Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks,
)

## 7. (Optional) Fine-tune — unfreeze top layers
This squeezes out extra accuracy by letting the last layers of MobileNetV2 adapt to leaf images specifically. Mention this technique in your report — it shows depth of understanding.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

fine_tune_history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=callbacks,
)

## 8. Evaluate — Confusion Matrix & Classification Report

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

val_gen.reset()
preds = model.predict(val_gen)
y_pred = np.argmax(preds, axis=1)
y_true = val_gen.classes

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Greens')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — include this in your report!')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 9. Plot Training Curves (for your report)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 10. Save Model + Class Labels

In [ ]:
import json

model.save('agrilens_model.keras')

with open('class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

print('✅ Model saved as agrilens_model.keras')
print('✅ Class names saved as class_names.json')
print('\nDownload both files from the Colab file panel (left sidebar) and place them in:')
print('agrilens/ml-model/saved_model/')

In [ ]:
from google.colab import files
files.download('agrilens_model.keras')
files.download('class_names.json')
files.download('confusion_matrix.png')
files.download('training_curves.png')

## 11. Quick test — predict on a single image

In [ ]:
from tensorflow.keras.preprocessing import image

def predict_image(img_path):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    pred = model.predict(img_array)[0]
    top_idx = np.argmax(pred)

    print(f'Prediction: {CLASS_NAMES[top_idx]}')
    print(f'Confidence: {pred[top_idx]*100:.2f}%')
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'{CLASS_NAMES[top_idx]} ({pred[top_idx]*100:.1f}%)')
    plt.show()

# Example usage (uncomment and provide a path):
# predict_image('/content/filtered_dataset/Tomato_Early_blight/some_image.jpg')